In [ ]:
# Lab type: write
# Course: DS101 — Introduction to Data Science
# Lesson: Exploratory Data Analysis
# Task: Run a full EDA checklist on an employee records dataset

# Lab: Exploratory Data Analysis

In this lab you'll run a full EDA checklist on a employee records dataset.
Work through each section in order — the questions at the end of each step
will guide your thinking.

**No sample answers are provided.** The goal is to practise asking questions
of a dataset you haven't seen before, not to arrive at a specific result.

## Step 1: Load and inspect the data

In [ ]:
import pandas as pd

# Load the employee records dataset
url = "https://raw.githubusercontent.com/SophiArch/notebooks/main/datasets/employee_records.csv"
df = pd.read_csv(url)

# Check the shape
print(df.shape)

# Check the column types
print(df.dtypes)

In [ ]:
# Preview the first 5 rows
df.head()

> **Question:** How many rows and columns does the dataset have?
> What data types do you see? Are any columns stored as the wrong type?

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Shape:** Call `df.shape` to get (rows, columns). For this dataset expect several hundred rows and around 8–12 columns covering employee demographics, salary, and tenure.

**Types check:** Use `df.dtypes` or `df.info()`. Common mismatch: a salary column stored as `object` if it was loaded with currency symbols or commas — fix with `df["salary"] = pd.to_numeric(df["salary"].str.replace(",", ""), errors="coerce")`. Date columns often arrive as `object`; cast them with `pd.to_datetime()`.

</details>

## Step 2: Missing values

In [ ]:
# Count missing values per column
print(df.isnull().sum())

> **Question:** Which columns have missing values? How would you handle each one?
> (Drop the row? Fill with a default? Leave it for now?)

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Strategy by column type:**
- *Numeric* (salary, tenure): fill missing values with the **median** rather than the mean — the median is unaffected by outliers that can distort the mean.
- *Categorical* (department, job title): fill with `"Unknown"` or the column mode (`df["col"].mode()[0]`), depending on whether the missing data carries meaning.
- *Drop rows* only when the majority of fields are absent; dropping on a single missing column can introduce selection bias into your analysis.

Run `df.isnull().mean()` to see the missing-value rate per column — anything above 30–40% deserves a separate discussion.

</details>

## Step 3: Category distributions

In [ ]:
# Use value_counts to explore each categorical column
# Identify the categorical columns first
cat_cols = df.select_dtypes(include="object").columns
print(cat_cols.tolist())

# Then explore each one
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

> **Question:** Are any categories heavily imbalanced? Are there any unexpected values
> (typos, inconsistent casing, `Unknown` entries)?

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Imbalance:** Use `value_counts(normalize=True)` to check proportions. If one category holds >80% of rows, group-level statistics for minority categories will be unreliable.

**Unexpected values:** Look for string variants of null: `"None"`, `"none"`, `"N/A"`, `"na"` — these won't be caught by `isnull()`. Normalise with `.str.lower().str.strip()` before counting, then replace known placeholders: `df.replace({"none": None, "n/a": None})`.

Inconsistent casing (e.g., `"Sales"` vs `"sales"`) will split what should be one category into two separate groups in your counts.

</details>

## Step 4: Numeric distributions

In [ ]:
# Summary statistics for all numeric columns
df.describe().round(2)

> **Question:** Do the min/max values look reasonable for each column?
> Is the mean much higher than the median (50th percentile)? That often signals outliers.

<details>
<summary>🔑 Reveal answer — Q4</summary>

**Reasonableness check:** Negative values in age, salary, or tenure indicate data entry errors. A salary of 0 is almost always a missing-value placeholder, not a real value.

**Mean vs median:** If `mean` >> `median` (the 50th percentile) for a column like salary, a small number of very high earners are pulling the mean up. The median is the better "typical employee" figure. A ratio of mean/median > 1.5 is a strong signal of right-skewed data worth investigating.

</details>

## Step 5: Group differences

In [ ]:
# Choose a numeric column and group it by a categorical column
# Replace 'sales_amount' and 'category' with the actual column names you found above
numeric_col = "salary"        # <-- change this
group_col = "department"    # <-- change this

print(
    df.groupby(group_col)[numeric_col]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
    .round(2)
)

> **Question:** Which group has the highest mean? Is the mean meaningfully different from
> the median within any group? (Large mean–median gaps suggest skewed distributions or outliers.)

<details>
<summary>🔑 Reveal answer — Q5</summary>

**Interpreting group differences:** Compare both mean and median across groups. If the group rankings change between the two statistics, the mean result is being driven by outliers — report the median as the primary figure.

**Mean–median gap within a group:** A large gap means the group's distribution is right-skewed, likely due to a few extreme values. A boxplot (`df.boxplot(column="salary", by="department")`) visualises the spread alongside the central tendency and makes outliers visible immediately.

</details>

## Step 6: Correlation

In [ ]:
# Correlation matrix for numeric columns
numeric_df = df.select_dtypes(include="number")
print(numeric_df.corr().round(2))

> **Question:** Which pair of columns is most strongly correlated?
> Is there any correlation above 0.9 or below -0.9?
> What might that mean for building a predictive model?

<details>
<summary>🔑 Reveal answer — Q6</summary>

**Near-perfect correlation (|r| > 0.9):** Means the two columns contain almost identical information. Including both in a predictive model causes multicollinearity — the model becomes numerically unstable and coefficients lose interpretability. Keep the one that's easier to measure or more directly meaningful; drop the other.

**Interpretation limits:** Correlation measures only *linear* relationships. A correlation near zero doesn't mean two variables are unrelated — they may have a strong non-linear relationship. And correlation says nothing about causation.

</details>

## Step 7: Write your findings

In the cell below, write 3–5 bullet points summarising the most important things
you discovered about this dataset. Imagine you're briefing a colleague who hasn't
seen the data.

*(Write your findings here.)*

- 
- 
- 

<details>
<summary>🔑 Model example findings</summary>

Example of what strong 3–5 bullet findings look like for an employee records EDA:

- The dataset contains 500 rows and 10 columns; `start_date` was stored as object and needed casting to datetime.
- Three columns have missing values: `salary` (4%), `department` (1%), `years_at_company` (2%) — filled with median/mode respectively.
- `department` is heavily imbalanced: Engineering (62% of staff), with all other departments under 15%.
- `salary` is right-skewed (mean £58k vs median £51k), indicating a minority of high earners; tenure shows a similar pattern.
- `salary` and `years_at_company` are correlated at r = 0.74 — longer-tenured employees earn more on average, though causation can't be assumed.

</details>